In [1]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)

print(f"La root del progetto è: {PROJECT_ROOT}")

La root del progetto è: /home/cvalentino/SissaUnisaDraftCodes/


In [2]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"

Import delle librerie

In [3]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

Fissiamo precisione doppia

In [4]:
torch.set_default_dtype(torch.float64)

# Problema parametrico in considerazione

Consideriamo il problema parametrico sul dominio $\Omega$ di frontiera $\Gamma = \partial \Omega$
$$
\begin{equation}
    \begin{cases}
        \frac{\partial^2 u}{\partial x^2} \left(x, y, z\right) + \frac{\partial^2 u}{\partial y^2} \left(x, y, z \right) + \frac{\partial^2 u}{\partial z^2} \left(x, y, z\right) = - \left( \alpha^2 + \beta^2 \right) \pi^2 \lambda x \cos\left( \alpha \pi y\right) \sin\left( \beta \pi z\right) & \left(x, y, z \right) \in \Omega \\
        u \left( x, y, z \right) = \lambda x \cos\left( \alpha \pi y \right) \sin \left( \beta \pi z \right) & \left( x, y, z \right) \in \Gamma
    \end{cases}
    \tag{1}
\end{equation}
$$

di soluzione analitica
$$
\begin{equation}
    u \left( x, y, z \right) = \lambda x \cos\left( \alpha \pi y \right) \sin \left( \beta \pi z \right) \qquad \left( x, y, z \right) \in \bar{\Omega}
    \tag{2}
\end{equation}
$$

## Creazione dei dati per il problema inverso

Fissiamo i punti in cui sono installati i sensori

In [5]:
rock = Blend2Pina(LOAD_MODEL + model_name)

Read blend: "/home/michael/Documenti/GithubProjects/SissaUnisaDraftCodes/models/rock/Rock1.blend"


Acquisizione dei punti al contrno

In [6]:
num_points = 500

surface = rock.boundary()
points = surface.sample(num_points)

Fissimao i parametri e collezioniamo i dati simulati

In [7]:
par_lambda = .1
par_alpha = .2
par_beta = .5

u = par_lambda * points.extract("x").tensor * torch.cos(par_alpha * torch.pi * points.extract("y").tensor) * torch.sin(par_beta * torch.pi * points.extract("z").tensor)

Creazione file .csv per conservare i punti al contorno

In [8]:
total_info = torch.concat(
    [points.tensor, u],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "u"]
)

df.to_csv("./files/data.csv", sep=";")

## Creazione mesh e xdmf

In [9]:
rock_msh = Blend2Mesh(LOAD_MODEL + model_name, "rock")

Read blend: "/home/michael/Documenti/GithubProjects/SissaUnisaDraftCodes/models/rock/Rock1.blend"


In [10]:
rock_msh.create_mesh(len_msh=0.07)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 10%] Meshing curve 3 (Line)
Info    : [ 10%] Meshing curve 4 (Line)
Info    : [ 10%] Meshing curve 5 (Line)
Info    : [ 10%] Meshing curve 6 (Line)
Info    : [ 10%] Meshing curve 7 (Line)
Info    : [ 10%] Meshing curve 8 (Line)
Info    : [ 10%] Meshing curve 9 (Line)
Info    : [ 10%] Meshing curve 10 (Line)
Info    : [ 10%] Meshing curve 11 (Line)
Info    : [ 10%] Meshing curve 12 (Line)
Info    : [ 10%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 10%] Meshing curve 16 (Line)
Info    : [ 10%] Meshing curve 17 (Line)
Info    : [ 10%] Meshing curve 18 (Line)
Info    : [ 10%] Meshing curve 19 (Line)
Info    : [ 10%] Meshing curve 20 (Line)
Info    : [ 10%] Meshing curve 21 (Line)
Info    : [ 10%] Meshing curve 22 (Line)
Info    : [ 10%] Meshing curve 23 (Line)
Info    : [ 10%] Meshing curve 24 (Line)
I

In [11]:
rock_xdmf = Msh2Xdmf("rock.msh", "rock")
rock_xdmf.to_xdmf()

Info    : Reading 'rock.msh'...
Info    : 867 entities
Info    : 42343 nodes
Info    : 263548 elements
Info    : Done reading 'rock.msh'                                          
Info    : Meshing 1D...
Info    : Done meshing 1D (Wall 0.000135249s, CPU 0.000158s)
Info    : Meshing 2D...
Info    : Done meshing 2D (Wall 9.1794e-05s, CPU 0.000112s)
Info    : Meshing 3D...
Info    : Done meshing 3D (Wall 0.355412s, CPU 0.355795s)
Info    : Optimizing mesh...
Info    : Done optimizing mesh (Wall 0.0131181s, CPU 0.013392s)
Info    : 42343 nodes 263548 elements
